In [3]:
import pandas as pd
import numpy as np
import random
import time

# Read input data
data = pd.read_excel('data/input42u.xlsx')
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# Read power demand data
power_demand_data = pd.read_excel('data/power_demand42u.xlsx')
power_demand = power_demand_data['load'].values

# WOA Parameters
num_whales = 30
num_iterations = 100

# Define WOA-based power calculation
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown):
    num_units = len(pmin)

    # Initialize whales (random solutions)
    whales = np.random.uniform(pmin, pmax, (num_whales, num_units))
    best_whale = np.zeros(num_units)
    best_score = float('inf')

    for iteration in range(num_iterations):
        # Update parameter a linearly
        a_param = 2 - 2 * (iteration / num_iterations)

        for i in range(num_whales):
            # Ensure whales respect generation limits
            whales[i] = np.clip(whales[i], pmin, pmax)

            # Calculate fuel cost for the whale
            squared_P = np.multiply(whales[i], whales[i])
            fuel_cost = np.sum(np.add(a, np.add(np.multiply(b, whales[i]), np.multiply(c, squared_P))))

            # Penalty for demand mismatch
            power_output = np.sum(whales[i])
            mismatch_penalty = abs(demand - power_output) * 10000  # Large penalty for mismatch
            score = fuel_cost + mismatch_penalty

            # Update the best whale
            if score < best_score:
                best_score = score
                best_whale = whales[i].copy()

        # Update whales' positions
        for i in range(num_whales):
            r = np.random.rand()
            if r < 0.5:
                # Encircling prey
                A = 2 * a_param * np.random.rand() - a_param
                C = 2 * np.random.rand()
                D = abs(C * best_whale - whales[i])
                whales[i] = best_whale - A * D
            else:
                # Spiral updating position
                b = 1  # Spiral parameter
                l = np.random.uniform(-1, 1)  # Random number in [-1, 1]
                distance_to_best = abs(best_whale - whales[i])
                whales[i] = distance_to_best * np.exp(b * l) * np.cos(2 * np.pi * l) + best_whale

            # Ensure whales respect generation limits
            whales[i] = np.clip(whales[i], pmin, pmax)

        # Normalization to meet demand
        total_power = np.sum(best_whale)
        if total_power != 0:
            scaling_factor = demand / total_power
            best_whale = best_whale * scaling_factor

        # Ensure best whale respects generation limits
        best_whale = np.clip(best_whale, pmin, pmax)

        # Check for ramp rate violations and correct
        if iteration > 0:
            diff = best_whale - whales[i]
            ramp_up_mask = diff > 0
            ramp_down_mask = diff < 0
            ramp_up_values = np.minimum(diff[ramp_up_mask], rampup[ramp_up_mask])
            ramp_down_values = np.maximum(diff[ramp_down_mask], -rampdown[ramp_down_mask])
            best_whale[ramp_up_mask] = whales[i][ramp_up_mask] + ramp_up_values
            best_whale[ramp_down_mask] = whales[i][ramp_down_mask] + ramp_down_values

        # Termination condition for power balance
        if abs(demand - np.sum(best_whale)) < 0.0001:
            break

    # Return the best solution
    best_power = best_whale
    return best_power, iteration + 1

# Fuel cost calculation
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.divide(np.add(a, np.add(np.multiply(b, P), np.multiply(c, squaredP))), 1000)
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# Loop through power demand data and calculate results
all_outputs = []
schedulling = []

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values
    P, iterations = calculate_power(demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down)
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    # Output Results
    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost: Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({'Demand': demand, 
                        'Output': output, 
                        'Total Power Produced': np.sum(P),
                        'Total Fuel Cost': total_Fuel_Cost,  # No need to sum again, it's already a single value
                        'Computation Time': end_time - start_time})

    # Scheduling
    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P[unit_index]
        output_info[f'Unit {unit} Cost'] = total_Fuel_Cost  # Assign total cost directly
    schedulling.append(output_info)

# Calculate ramp rates and save to an additional sheet
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):  # Start from 1 because there's no previous output for the first demand
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    # Check for ramp rate violations
    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = violations[unit_index]
               
        # Check for generation limit violations
        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = gen_violation
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)
# Save results to Excel
output_file_path = 'WOA (ED).xlsx'
with pd.ExcelWriter(output_file_path) as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(all_outputs).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)
print("All results saved to 'WOA (ED).xlsx'")

Load Counter: 1
Output for Power Demand 8761.42 MW
    Unit  Power Produced (MW)
0      1           285.239745
1      2           285.239745
2      3           285.239745
3      4           285.239745
4      5           465.511263
5      6           465.511263
6      7           465.511263
7      8            99.833911
8      9            99.833911
9     10            99.833911
10    11            99.833911
11    12            45.638359
12    13            45.638359
13    14            85.571923
14    15            85.571923
15    16           251.010975
16    17           251.010975
17    18           251.010975
18    19             5.704795
19    20             5.704795
20    21            68.457539
21    22            68.457539
22    23            68.457539
23    24            68.457539
24    25            68.457539
25    26            68.457539
26    27            68.457539
27    28            68.457539
28    29           456.383592
29    30           456.383592
30    31           